In [11]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))
import mrs
import numpy as np
import pickle

In [12]:
# load example problem and Pareto front
DATA_DIR = "../simulations"

SETUP = {
    "n_obj": 5,
    "n_con": 1,
    "n_items": 100,
    "n_selected": 25,
    "rho": 0.2,
    "exactly": True,
    "aspiration": np.array([14582, 11126,  7081,  9801,  8849]),
    "obj_names": ["obj_1", "obj_2", "obj_3", "obj_4", "obj_5"]
}

data = np.load(f'{DATA_DIR}/problems/items_{SETUP["n_obj"]}obj_{SETUP["n_items"]}items_{SETUP["rho"]}rho_allneg.npz')
items = data['items']
capacity = data['capacity']

RESULTS_DIR = "exact" if SETUP["exactly"] else "upper_bound"
SUB_DIR = "prefer" if SETUP["aspiration"] is not None else "no_prefer"
with open(f"{DATA_DIR}/results/{RESULTS_DIR}/{SUB_DIR}/eda_mode2_{SETUP['n_obj']}obj_{SETUP['n_items']}items_{SETUP['rho']}rho_allneg_withtime.pkl", "rb") as f:
    results = pickle.load(f)
pf_raw = results["converged_pf_table"][-1]
ps_raw = results["converged_pf_population_table"][-1]

In [ ]:
# compute mrs
OUTPUT_DIR = f"results/{SETUP["n_obj"]}obj_{SETUP["n_items"]}items_{SETUP["rho"]}rho"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
OBJ_COLORS = {
    "obj_1": "#4C78A8",   # blue
    "obj_2": "#F58518",   # orange
    "obj_3": "#54A24B",   # green
    "obj_4": "#E45756",   # red
    "obj_5": "#B279A2",   # purple
    "obj_6": "#FF9DA6",   # pink
    "obj_7": "#9D755D",   # brown
    "obj_8": "#2CB1BC",   # teal
}
density_threshold = 0.8

pf, kept_index = mrs.density_filter(pf_raw, density_threshold)
df, _ = mrs.pf_to_csv(pf, OUTPUT_DIR, SETUP["obj_names"])
mrs_params = mrs.get_mrs_params()
info = mrs.get_pf_info(df)
query_idx, query_pt = mrs.get_query_point(info["data"], SETUP["aspiration"], info["iqr"])
betas_raw, betas = mrs.compute_mrs_at_query(
        info["data"], 
        mrs_params["k"], 
        info["d"], 
        info["iqr"], 
        query_idx
    )
mrs.save_mrs_results(betas_raw, betas, OUTPUT_DIR, info["obj_names"])
mrs.plot_bar_chart(
        betas, 
        info["obj_names"], 
        query_pt, 
        info["d"],
        OUTPUT_DIR, 
        obj_colors=OBJ_COLORS,
    )
radii_pct, radii_raw = mrs.compute_validity_radii(
        info["data"], 
        mrs_params["k"], 
        info["d"], 
        info["iqr"],
        query_idx, 
        mrs_params["epsilon"],
    )
mrs.plot_validity_bar_chart(
    radii_pct, 
    radii_raw, 
    info["obj_names"], 
    info["d"], 
    mrs_params["epsilon"],
    OUTPUT_DIR, 
    obj_colors=OBJ_COLORS,
)

Saved 7628 points → results/5obj_100items_0.2rho/pf.csv


PosixPath('results/5obj_100items_0.2rho/validity_bar_chart.png')

In [15]:
print(SETUP["aspiration"])
print(query_pt)

[14582 11126  7081  9801  8849]
[14737. 11515.  7142.  9902.  8979.]


In [16]:
from scipy.stats import percentileofscore
obj_weight = np.array([
    percentileofscore(pf[:,i], SETUP["aspiration"][i])
    for i in range(pf.shape[1])
])
print(obj_weight)

[67.57341374 59.025957    8.85553225 30.5715784  33.91452543]
